# Day 40 · The training loop: forward, backward, update, repeat

**First-principles question:** What actually happens when a model 'trains'?

The full task (learn → build → record) is in this folder's `README.md`. This notebook is your workbench:
run cells top to bottom with **Shift + Enter**, fill in every cell marked **TODO**, and finish with the
**✅ self-check** cell at the end.

In [ ]:
import sys
from pathlib import Path

def find_root():
    """The fde-first-principles folder (it has progress.json)."""
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p / "progress.json").exists():
            return p
    raise RuntimeError("Open this notebook from inside your fde-first-principles folder.")

ROOT = find_root()
sys.path.insert(0, str(ROOT))  # so `import lab.…` works

import importlib
import math
import lab.micrograd as micrograd
importlib.reload(micrograd)          # picks up your latest edits to lab/micrograd.py
from lab.micrograd import Value, draw
import random
import numpy as np
import matplotlib.pyplot as plt
import lab.nn as nn
importlib.reload(nn)
from lab.nn import MLP
random.seed(1337)

## 1 · The training loop

In [ ]:
xs = [[2.0, 3.0, -1.0], [3.0, -1.0, 0.5], [0.5, 1.0, 1.0], [1.0, 1.0, -1.0]]
ys = [1.0, -1.0, -1.0, 1.0]
model = MLP(3, [4, 4, 1])
LEARNING_RATE, STEPS = 0.05, 100
losses = []

for step in range(STEPS):
    # TODO 1 forward:  predictions for every x, loss = sum of (prediction - y) ** 2  (start the sum at Value(0.0))
    # TODO 2 zero:     set p.grad = 0.0 for every p in model.parameters()   ← try once WITHOUT this line
    # TODO 3 backward: loss.backward()
    # TODO 4 update:   p.data -= LEARNING_RATE * p.grad for every parameter
    # TODO 5 record:   losses.append(loss.data)
    pass

plt.plot(losses); plt.yscale("log"); plt.xlabel("step"); plt.ylabel("loss"); plt.show()
print("final predictions:", [round(model(x).data, 3) for x in xs])

**What happened when you left out the zero step?** …

## 2 · The escalation data from project 3
Same 300 customers. Labels −1 (no) and +1 (yes) to match tanh.

In [ ]:
rng = np.random.default_rng(11)
days_waiting = rng.uniform(0, 5, 300).round(1)
complaints = rng.poisson(1.0, 300)
true_z = -4.0 + 1.0 * days_waiting + 0.9 * complaints
escalated = rng.random(300) < 1 / (1 + np.exp(-true_z))
X = [[d / 5, c / 5] for d, c in zip(days_waiting, complaints)]       # scaled to about 0..1
Y = [1.0 if e else -1.0 for e in escalated]

net = MLP(2, [4, 1])
for step in range(60):
    preds = [net(x) for x in X[:200]]
    loss = sum(((p - y) ** 2 for p, y in zip(preds, Y[:200])), Value(0.0)) * (1 / 200)
    for p in net.parameters():
        p.grad = 0.0
    loss.backward()
    for p in net.parameters():
        p.data -= 0.5 * p.grad
test_accuracy = np.mean([(net(x).data > 0) == (y > 0) for x, y in zip(X[200:], Y[200:])])
print(f"test accuracy {test_accuracy:.0%} (project 3's logistic regression: see your day-8 notebook)")

In [ ]:
# ✅ Self-check: run me last. A clear message tells you what to fix.
assert len(losses) == STEPS, "Step 1: record the loss every step"
assert losses[-1] < 0.05, f"The loss should get below 0.05 (it is {losses[-1]:.3f}). Did you zero the grads?"
assert test_accuracy > 0.6
print("All checks passed ✅  Now write 'In my words' in README.md and record your video.")